# Урок 12. Decision Trees — деревья решений
**Библиотеки:** numpy, sklearn, matplotlib

**Цель:** понять, как дерево задаёт вопросы и почему легко переобучается.

## Простыми словами
Дерево — это игра «20 вопросов»: «Площадь > 70? → да → Комнат > 2? → да → дорого».
Каждый узел — вопрос про один признак, листья — ответы.

Как дерево выбирает вопросы? Перебирает варианты и берёт тот, который лучше всего
разделяет классы — делает группы «чище». Чистоту меряют **Gini** или **entropy**:
0 = в группе один класс (идеально чисто), максимум = классы 50/50 (полный хаос).

Плюсы: не нужен scaling, понятная логика, ловит нелинейности.
Минус: без ограничения глубины зубрит данные до листа на каждый пример.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

# Студенты: часы сна и часы подготовки -> сдал/нет
rng = np.random.default_rng(8)
n = 300
sleep = rng.uniform(3, 9, n)
study = rng.uniform(0, 10, n)
passed = ((study > 4) & (sleep > 5) | (study > 8)).astype(int)
passed ^= (rng.uniform(0, 1, n) < 0.07).astype(int)   # 7% шума — жизнь несправедлива

X = np.column_stack([sleep, study])
X_tr, X_te, y_tr, y_te = train_test_split(X, passed, test_size=0.3, random_state=42)

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
print("train acc:", tree.score(X_tr, y_tr).round(3), "| test acc:", tree.score(X_te, y_te).round(3))

plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=['сон', 'учёба'], class_names=['провал', 'сдал'],
          filled=True, fontsize=9)
plt.show()

In [ ]:
# Граница решений: дерево режет пространство прямоугольниками
xx, yy = np.meshgrid(np.linspace(3, 9, 200), np.linspace(0, 10, 200))
for depth in [3, None]:   # None = без ограничений
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_tr, y_tr)
    Z = t.predict(np.column_stack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
    plt.scatter(X[:, 0], X[:, 1], c=passed, cmap='RdYlGn', s=12, edgecolors='k', linewidths=0.3)
    plt.xlabel('сон'); plt.ylabel('учёба')
    plt.title(f'depth={depth}: train={t.score(X_tr,y_tr):.2f} test={t.score(X_te,y_te):.2f}')
    plt.show()

## Что произошло
plot_tree показал логику: дерево само нашло правила, близкие к настоящим («учёба > ~4 и сон > ~5»).
На карте границ: depth=3 — крупные разумные прямоугольники; depth=None — модель вырезала
закуток под каждую шумную точку (train=1.0, test просел) — классический overfit.

## Типичные ошибки
1. Не ограничивать глубину (max_depth) — почти гарантированный overfit.
2. Масштабировать признаки для дерева — не вредно, но бессмысленно.
3. Доверять одному дереву — оно нестабильно: чуть другие данные → другое дерево. Решение — лес (урок 13).

## Конспект 📓
Дерево = цепочка if-вопросов. Gini/entropy = мера «грязности» группы, дерево минимизирует её.
max_depth — главный тормоз переобучения. Scaling не нужен. plot_tree — посмотреть логику.

---
## Мини-задание 💪
Обучи DecisionTreeClassifier с min_samples_leaf=20 (лист не меньше 20 примеров) без ограничения depth. Помогло ли против переобучения? Сравни test accuracy.

## 5 проверочных вопросов ❓
1. Как дерево принимает решение для нового примера?
2. Что такое Gini простыми словами?
3. Почему дерево без max_depth переобучается?
4. Нужен ли деревьям feature scaling?
5. Как выглядит граница решений дерева на 2D-графике?

## Домашнее задание 🏠
Задача 24 из practice_tasks.md (если не делал). Выведи tree.feature_importances_ — какой признак важнее, сон или учёба?
